In [2]:
import pandas as pd

df = pd.read_csv('data/Sample - Superstore.csv', encoding='latin1')
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   str    
 2   Order Date     9994 non-null   str    
 3   Ship Date      9994 non-null   str    
 4   Ship Mode      9994 non-null   str    
 5   Customer ID    9994 non-null   str    
 6   Customer Name  9994 non-null   str    
 7   Segment        9994 non-null   str    
 8   Country        9994 non-null   str    
 9   City           9994 non-null   str    
 10  State          9994 non-null   str    
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   str    
 13  Product ID     9994 non-null   str    
 14  Category       9994 non-null   str    
 15  Sub-Category   9994 non-null   str    
 16  Product Name   9994 non-null   str    
 17  Sales          9994 non-null   float64
 18  Quantity       9994

In [4]:
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%m/%d/%Y')

df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Shipping Days'] = (df['Ship Date'] - df['Order Date']).dt.days

df[['Order Date', 'Ship Date', 'Order Year', 'Order Month', 'Shipping Days']].head()

,Order Date,Ship Date,Order Year,Order Month,Shipping Days
0,2016-11-08,2016-11-11,2016,11,3
1,2016-11-08,2016-11-11,2016,11,3
2,2016-06-12,2016-06-16,2016,6,4
3,2015-10-11,2015-10-18,2015,10,7
4,2015-10-11,2015-10-18,2015,10,7


In [5]:
df[['Sales', 'Profit', 'Discount', 'Quantity']].describe()

,Sales,Profit,Discount,Quantity
count,9994.000000,9994.000000,9994.000000,9994.000000
mean,229.858001,28.656896,0.156203,3.789574
std,623.245101,234.260108,0.206452,2.225110
min,0.444000,-6599.978000,0.000000,1.000000
25%,17.280000,1.728750,0.000000,2.000000
50%,54.490000,8.666500,0.200000,3.000000
75%,209.940000,29.364000,0.200000,5.000000
max,22638.480000,8399.976000,0.800000,14.000000


In [6]:
df['Profit Margin'] = df['Profit'] / df['Sales']

In [7]:
df[['Sales', 'Profit', 'Profit Margin']].describe()

,Sales,Profit,Profit Margin
count,9994.000000,9994.000000,9994.000000
mean,229.858001,28.656896,0.120314
std,623.245101,234.260108,0.466754
min,0.444000,-6599.978000,-2.750000
25%,17.280000,1.728750,0.075000
50%,54.490000,8.666500,0.270000
75%,209.940000,29.364000,0.362500
max,22638.480000,8399.976000,0.500000


In [8]:
import sqlite3

conn = sqlite3.connect('data/superstore.db')
df.to_sql('sales', conn, if_exists='replace', index=False)

9994

In [9]:
query = """
SELECT Category, SUM(Sales) as Total_Sales, SUM(Profit) as Total_Profit
FROM sales
GROUP BY Category
"""

result = pd.read_sql_query(query, conn)
result

,Category,Total_Sales,Total_Profit
0,Furniture,741999.7953,18451.2728
1,Office Supplies,719047.0320,122490.8008
2,Technology,836154.0330,145454.9481


In [10]:
query = """
SELECT Category, 
       SUM(Sales) as Total_Sales, 
       SUM(Profit) as Total_Profit,
       AVG(Discount) as Avg_Discount
FROM sales
GROUP BY Category
"""

result = pd.read_sql_query(query, conn)
result

,Category,Total_Sales,Total_Profit,Avg_Discount
0,Furniture,741999.7953,18451.2728,0.173923
1,Office Supplies,719047.0320,122490.8008,0.157285
2,Technology,836154.0330,145454.9481,0.132323


In [11]:
query = """
SELECT [Sub-Category], 
       SUM(Sales) as Total_Sales, 
       SUM(Profit) as Total_Profit,
       AVG(Discount) as Avg_Discount
FROM sales
WHERE Category = 'Furniture'
GROUP BY [Sub-Category]
ORDER BY Total_Profit ASC
"""

result = pd.read_sql_query(query, conn)
result

,Sub-Category,Total_Sales,Total_Profit,Avg_Discount
0,Tables,206965.5320,-17725.4811,0.261285
1,Bookcases,114879.9963,-3472.5560,0.211140
2,Furnishings,91705.1640,13059.1436,0.138349
3,Chairs,328449.1030,26590.1663,0.170178


In [12]:
query = """
SELECT [Order ID], [Product Name], Sales, Discount, Profit
FROM sales
WHERE [Sub-Category] = 'Tables'
ORDER BY Profit ASC
LIMIT 15
"""

result = pd.read_sql_query(query, conn)
result

,Order ID,Product Name,Sales,Discount,Profit
0,CA-2015-116638,Chromcraft Bull-Nose Wood Oval Conference Tabl...,4297.644,0.40,-1862.3124
1,US-2017-162558,Chromcraft Bull-Nose Wood Oval Conference Tabl...,2314.116,0.40,-1002.7836
2,CA-2017-159100,Balt Solid Wood Round Tables,1875.258,0.40,-968.8833
3,CA-2016-109869,Bush Advantage Collection Racetrack Conference...,1272.630,0.50,-814.4832
4,CA-2014-108609,Hon 94000 Series Round Tables,1421.664,0.40,-734.5264
5,CA-2017-142090,Bush Advantage Collection Racetrack Conference...,1781.682,0.40,-653.2834
6,US-2017-110576,"Riverside Furniture Oval Coffee Table, Oval En...",2065.320,0.40,-619.5960
7,US-2017-124968,BoxOffice By Design Rectangular and Half-Moon ...,765.625,0.50,-566.5625
8,US-2014-148838,Balt Solid Wood Round Tables,1071.576,0.40,-553.6476
9,CA-2016-168032,Bretford Just In Time Height-Adjustable Mult...,626.100,0.50,-538.4460


In [13]:
query = """
SELECT 
    CASE 
        WHEN Discount = 0 THEN '0%'
        WHEN Discount <= 0.2 THEN '1-20%'
        WHEN Discount <= 0.4 THEN '21-40%'
        ELSE '41%+'
    END as Discount_Range,
    COUNT(*) as Num_Orders,
    SUM(Profit) as Total_Profit,
    AVG(Profit) as Avg_Profit
FROM sales
WHERE [Sub-Category] = 'Tables'
GROUP BY Discount_Range
ORDER BY Discount_Range
"""

result = pd.read_sql_query(query, conn)
result

,Discount_Range,Num_Orders,Total_Profit,Avg_Profit
0,0%,72,13276.2997,184.393051
1,1-20%,71,-303.5580,-4.275465
2,21-40%,129,-19589.7244,-151.858329
3,41%+,47,-11108.4984,-236.351030


In [14]:
df.to_csv('data/superstore_clean.csv', index=False)
print("Esportato dataset pulito per Tableau")

Esportato dataset pulito per Tableau


In [15]:
query = """
SELECT [Order Year] as Year,
       SUM(Sales) as Total_Sales,
       SUM(Profit) as Total_Profit,
       ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) as Margin_Pct
FROM sales
GROUP BY Year
ORDER BY Year
"""

pd.read_sql_query(query, conn)

,Year,Total_Sales,Total_Profit,Margin_Pct
0,2014,484247.4981,49543.9741,10.23
1,2015,470532.5090,61618.6037,13.10
2,2016,609205.5980,81795.1743,13.43
3,2017,733215.2552,93439.2696,12.74


In [16]:
query = """
SELECT [Order Year] as Year,
       ROUND(AVG(Discount), 4) as Avg_Discount,
       ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) as Margin_Pct
FROM sales
GROUP BY Year
ORDER BY Year
"""

pd.read_sql_query(query, conn)

,Year,Avg_Discount,Margin_Pct
0,2014,0.1583,10.23
1,2015,0.1556,13.10
2,2016,0.1547,13.43
3,2017,0.1565,12.74


In [17]:
query = """
SELECT [Order Year] as Year, Category,
       ROUND(SUM(Sales), 0) as Total_Sales
FROM sales
GROUP BY Year, Category
ORDER BY Year, Category
"""

pd.read_sql_query(query, conn)

,Year,Category,Total_Sales
0,2014,Furniture,157193.0
1,2014,Office Supplies,151776.0
2,2014,Technology,175278.0
3,2015,Furniture,170518.0
4,2015,Office Supplies,137233.0
5,2015,Technology,162781.0
6,2016,Furniture,198901.0
7,2016,Office Supplies,183940.0
8,2016,Technology,226364.0
9,2017,Furniture,215387.0


In [18]:
query = """
SELECT Region, Category, 
       ROUND(SUM(Sales), 0) as Total_Sales,
       ROUND(SUM(Profit), 0) as Total_Profit,
       ROUND(AVG(Discount), 3) as Avg_Discount
FROM sales
WHERE Category = 'Furniture'
GROUP BY Region
ORDER BY Total_Profit ASC
"""

pd.read_sql_query(query, conn)

,Region,Category,Total_Sales,Total_Profit,Avg_Discount
0,Central,Furniture,163797.0,-2871.0,0.297
1,East,Furniture,208291.0,3046.0,0.154
2,South,Furniture,117299.0,6771.0,0.122
3,West,Furniture,252613.0,11505.0,0.131
